# 搜索与深度研究：在候选空间里寻找答案


> 第 6 讲用重复采样证明了"不动权重也能变强"，第 7 讲让智能体在开放环境中自我进化。两讲共同的思路是把计算从训练期搬到推理期，但都没有细究一个环节：生成了一大堆候选之后，如何在有限预算内挑出真正能用的那一个。
>
> 这一讲把"挑选"做成一条完整管线。我们先在程序空间里实现 AlphaCode 的采样、过滤、聚类流程，量化每一步筛掉多少候选；再给管线加上重排序器，观察更强的模型如何把所需采样量压缩几个数量级；随后进入知识空间，实现 Search-o1 的检索注入循环，让推理模型按需补上知识缺口；最后把各环节串成一个小型深度研究工作流。


"单次生成一个答案"不总够用，原因在任务本身。对编程竞赛题，答案是一个程序，必须全部通过隐藏测试，一次生成往往差一点；对开放知识问题，模型的记忆有边界，推理到一半就缺一段事实。两个问题的共同解法是搜索：在一个空间里生成大量候选，再依据信号把它们压缩到有限几个结果。

搜索的空间不同，骨架相同：生成候选，用廉价信号过滤，挑出值得提交或继续推理的部分。AlphaCode 在程序空间搜索，Search-o1 在知识空间搜索。本节先从一个具体任务出发，把程序空间里的搜索完整实现一遍。


## 1. 程序合成中的搜索

选一道 toy 编程任务：写一个函数，判断输入整数是否为质数，是则返回 True，否则返回 False。题面会给出几个输入输出对作为`样例测试`，提交系统用另一组`隐藏测试`判定对错。

**实验语料**：为模拟"模型采样出一批候选程序"，准备一个候选池。池子由脚本生成：三个行为正确的实现（写法不同），加六个刻意写错的变体，错误包括把 2 误判为合数、把 1 判为质数、检查范围出错等。管线唯一可见的正确性信号是样例测试，隐藏测试全程不可见。


In [ ]:
import numpy as np

np.random.seed(42)

# 候选程序池。真实管线里这里是模型采样出的源码字符串，经 exec 编译后执行；
# 为教学清晰，直接用函数对象表示候选程序。
def correct_naive(n):
    """朴素质数判断：从 2 遍历到 n-1。"""
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def correct_sqrt(n):
    """质数判断：只检查到根号 n。"""
    if n < 2:
        return False
    i = 2
    while i * i <= n:
        if n % i == 0:
            return False
        i += 1
    return True


def correct_div2(n):
    """质数判断：先处理偶数，奇数从 3 步进检查。"""
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    for i in range(3, n, 2):
        if n % i == 0:
            return False
    return True


def wrong_even_composite(n):
    """错误：把偶数一律判为合数，n=2 被误判。"""
    if n < 2:
        return False
    if n % 2 == 0:
        return False
    for i in range(3, n):
        if n % i == 0:
            return False
    return True


def wrong_one_is_prime(n):
    """错误：把 n=1 判为质数。"""
    if n == 1:
        return True
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def wrong_all_composite(n):
    """错误：从 1 开始检查，n % 1 == 0 恒成立，全判为合数。"""
    for i in range(1, n):
        if n % i == 0:
            return False
    return True


def wrong_trial_range(n):
    """错误：检查范围缺上界，n=4 这类合数被漏掉。"""
    for i in range(2, n // 2):
        if n % i == 0:
            return False
    return True


def wrong_mod_division(n):
    """错误：取模写成整除，n // i == 0 永不成立，全判为质数。"""
    for i in range(2, n):
        if n // i == 0:
            return False
    return True


def wrong_stop_early(n):
    """错误：循环上界少一个等号，平方数被误判。"""
    i = 2
    while i * i < n:
        if n % i == 0:
            return False
        i += 1
    return True


POOL = [
    {"name": "correct_naive", "fn": correct_naive, "good": True},
    {"name": "correct_sqrt", "fn": correct_sqrt, "good": True},
    {"name": "correct_div2", "fn": correct_div2, "good": True},
    {"name": "wrong_even_composite", "fn": wrong_even_composite, "good": False},
    {"name": "wrong_one_is_prime", "fn": wrong_one_is_prime, "good": False},
    {"name": "wrong_all_composite", "fn": wrong_all_composite, "good": False},
    {"name": "wrong_trial_range", "fn": wrong_trial_range, "good": False},
    {"name": "wrong_mod_division", "fn": wrong_mod_division, "good": False},
    {"name": "wrong_stop_early", "fn": wrong_stop_early, "good": False},
]
POOL_BY_NAME = {p["name"]: p for p in POOL}

# 样例测试：题面给出的输入输出对，过滤阶段唯一可见的正确性信号
EXAMPLE_TESTS = [(3, True), (4, False), (9, False), (13, True)]
# 隐藏测试：提交时才会揭晓，管线运行过程中不可见
HIDDEN_TESTS = [
    (1, False), (2, True), (5, True), (8, False), (11, True),
    (12, False), (17, True), (21, False), (23, True), (29, True),
]

n_good = sum(p["good"] for p in POOL)
print("候选池：%d 个程序，其中正确 %d 个、错误 %d 个"
      % (len(POOL), n_good, len(POOL) - n_good))
for p in POOL:
    print("  %-22s good=%s" % (p["name"], p["good"]))


先手算一遍`行为签名`的含义。取三个候选，在每个探测输入上执行，把输出记成一行 0/1 向量，这一行就是该程序的行为签名。签名相同的程序行为相同，会被归到同一个簇。


In [ ]:
# 手算行为签名：3 个候选 × 3 个探测输入
probe = [1, 2, 9]
picks = ["correct_sqrt", "wrong_even_composite", "wrong_one_is_prime"]
mat = np.array([[1 if POOL_BY_NAME[n]["fn"](x) else 0 for x in probe]
                for n in picks])
print("探测输入 :", probe)
print("每行是一个程序的行为签名（1=判为质数）：")
for i, n in enumerate(picks):
    print("  %-20s %s" % (n, mat[i].tolist()))
print("关键观察：签名互不相同的程序落在不同簇，签名相同的程序合并为一簇。")


第一步是采样。真实系统在推理时以较高温度从模型里采样出上百万个候选。这里用一个采样器模拟：给定质量参数 p_correct，每次以该概率采到某个正确实现，否则从错误变体里采一个。质量参数是模型能力的代理，模型越强，采到正确实现的概率越高。


In [ ]:
def sample_candidates(rng, n, p_correct=0.2):
    """模拟 LLM 采样：以 p_correct 的概率采到正确程序，否则采错误程序。

    n 为采样数量；质量参数 p_correct 是模型能力的代理，允许重复采样。
    """
    good = [p for p in POOL if p["good"]]
    bad = [p for p in POOL if not p["good"]]
    picked = []
    for _ in range(n):
        if rng.random() < p_correct:
            picked.append(rng.choice(good))
        else:
            picked.append(rng.choice(bad))
    return picked


rng = np.random.default_rng(42)
N = 300
cands = sample_candidates(rng, N, p_correct=0.2)
n_ok = sum(c["good"] for c in cands)
print("采样 %d 个候选，其中正确 %d 个（%.1f%%）"
      % (N, n_ok, 100.0 * n_ok / N))


第二步是过滤。验证器把候选跑在样例测试上，任一输入输出不匹配即剔除。样例测试是过滤阶段唯一合法的正确性信号，隐藏测试在这个阶段不可见。这一步的目标是用廉价信号把海量候选压到可管理的规模。


In [ ]:
def run_tests(fn, tests):
    """把程序跑在 (输入, 期望输出) 列表上，返回 (通过条数, 总条数)。"""
    ok = 0
    for x, want in tests:
        try:
            if bool(fn(x)) == want:
                ok += 1
        except Exception:
            pass
    return ok, len(tests)


def filter_by_example_tests(candidates, tests):
    """返回全部通过样例测试的候选。"""
    return [c for c in candidates if run_tests(c["fn"], tests)[0] == len(tests)]


passed = filter_by_example_tests(cands, EXAMPLE_TESTS)
n_passed_good = sum(c["good"] for c in passed)
print("过滤前 %d 个候选 -> 通过样例测试 %d 个（筛掉 %.1f%%）"
      % (len(cands), len(passed), 100.0 * (1 - len(passed) / len(cands))))
print("通过样例测试的候选中，正确的 %d 个，混入错误 %d 个"
      % (n_passed_good, len(passed) - n_passed_good))
print("关键观察：样例测试覆盖不全，两个错误变体混了进来。")
print("论文场景里样例测试能筛掉约 99%，本例任务简单、错误模式有限，比例低一些。")


第三步是聚类。过滤后剩下的候选仍可能超过提交配额，而且里面还混着样例测试没抓住的错误解。聚类不能用隐藏测试，改用一组额外的探测输入生成行为签名，把行为相同的候选归到一簇。正确解的行为彼此相似，会聚成一个大簇；错误解各错各的，散成许多小簇。选代表时从大簇开始，每簇取一个，覆盖尽可能多的不同解法。


In [ ]:
from collections import defaultdict


def behavior_signature(c, probe_inputs):
    """候选在 probe_inputs 上的输出元组，作为行为签名。"""
    return tuple(bool(c["fn"](x)) for x in probe_inputs)


def cluster_by_signature(candidates, probe_inputs):
    """按行为签名分组，返回按簇大小降序的簇列表，每簇为 (签名, [候选])。"""
    groups = defaultdict(list)
    for c in candidates:
        groups[behavior_signature(c, probe_inputs)].append(c)
    return sorted(groups.items(), key=lambda kv: len(kv[1]), reverse=True)


def submit_top_clusters(clusters, n_submit=10):
    """从最大的若干簇各取一个代表，作为提交。"""
    return [members[0] for _, members in clusters[:n_submit]]


def solves_hidden(c, tests):
    """候选能否全部通过隐藏测试，即这道题是否被它解出。"""
    return run_tests(c["fn"], tests)[0] == len(tests)


# 测试输入生成：边界值加随机整数，模拟 AlphaCode 的测试输入生成模型
probe_inputs = [0, 1, 2, 3, 4, 9, 16, 25, 100] + list(rng.integers(1, 300, size=20))
clusters = cluster_by_signature(passed, probe_inputs)
subs = submit_top_clusters(clusters)
n_solved = sum(solves_hidden(c, HIDDEN_TESTS) for c in subs)

print("过滤后 %d 个候选聚成 %d 个行为簇" % (len(passed), len(clusters)))
for i, (sig, members) in enumerate(clusters[:5]):
    n_g = sum(c["good"] for c in members)
    print("  簇 %d 大小 %3d，其中正确 %d" % (i, len(members), n_g))
print("提交 %d 个代表，其中真正解出隐藏测试的：%d 个" % (len(subs), n_solved))
print("汇总：采样 %d -> 过滤剩 %d（%.1f%%）-> 聚成 %d 簇 -> 提交 %d"
      % (len(cands), len(passed), 100.0 * len(passed) / len(cands),
         len(clusters), len(subs)))
print("关键观察：正确解聚成最大簇，聚类后的提交能解出隐藏测试。")


单道题能看到"更多采样是否换来更多解出"。在有限提交配额下，管线的质量由两个量衡量。`pass@k` 是所有 k 个候选全部提交时解出该题的概率，是理论上界；`10@k` 是过滤、聚类后只能提交 10 个时解出该题的概率。两者的差距正是挑选算法的损耗。AlphaCode 论文还有一个关键观察：求解率随采样数近似 log-linear 增长，更强的模型在更少的样本上达到同样的求解率。

下面把任务扩成四道 toy 题，实测无管线的随机提交、过滤加聚类的 10@k，并对照解析的 pass@k 上界。每道题有自己的候选池、样例测试与隐藏测试，行为签名的探测输入固定生成。


In [ ]:
def make_pool(goods, bads):
    """把正确变体与错误变体打包成候选池。"""
    pool = [{"name": "g%d" % i, "fn": f, "good": True} for i, f in enumerate(goods)]
    pool += [{"name": "b%d" % i, "fn": f, "good": False} for i, f in enumerate(bads)]
    return pool


def pow_loop(x):
    """2 的幂：反复除以 2 直到 1。"""
    if x <= 0:
        return False
    while x > 1:
        if x % 2 == 1:
            return False
        x //= 2
    return True


# 三道辅助题用 lambda 池，主演示题 is_prime 复用上面的 POOL。
# 每道题刻意留一个"样例放走、隐藏抓住"的错误变体。
PROBLEMS = [
    {"name": "is_prime",
     "pool": POOL,
     "examples": EXAMPLE_TESTS,
     "hidden": HIDDEN_TESTS},
    {"name": "is_square",
     "pool": make_pool(
         [lambda x: int(x ** 0.5) ** 2 == x,
          lambda x: any(i * i == x for i in range(x + 1))],
         [lambda x: any(i * i == x for i in range(x)),
          lambda x: int(x ** 0.5) ** 2 == x and x > 1]),
     "examples": [(1, True), (4, True), (9, True), (8, False)],
     "hidden": [(0, True), (16, True), (2, False), (25, True),
                (3, False), (100, True), (7, False)]},
    {"name": "is_power_of_two",
     "pool": make_pool(
         [lambda x: x > 0 and (x & (x - 1)) == 0, pow_loop],
         [lambda x: (x & (x - 1)) == 0,
          lambda x: x > 1 and (x & (x - 1)) == 0]),
     "examples": [(1, True), (2, True), (4, True), (6, False), (8, True)],
     "hidden": [(0, False), (1, True), (16, True), (3, False), (32, True),
                (10, False), (64, True), (5, False)]},
    {"name": "is_palindrome",
     "pool": make_pool(
         [lambda x: str(x) == str(x)[::-1],
          lambda x: all(str(x)[i] == str(x)[-1 - i]
                        for i in range(len(str(x)) // 2))],
         [lambda x: str(x) == str(x)[::-1] or x > 999,
          lambda x: str(x) == str(x)[::-1] and len(str(x)) % 2 == 1]),
     "examples": [(0, True), (121, True), (123, False), (12321, True), (10, False)],
     "hidden": [(1, True), (22, True), (123, False), (1221, True),
                (100, False), (345, False), (1231, False)]},
]

# 每道题用能区分正确与错误变体的探测输入，保证错误解与正确解不在同一簇
base_probes = [0, 1, 2, 3, 4, 9, 16, 25, 100]
extra_probes = {
    "is_prime": [2, 4, 9, 16, 25, 100],
    "is_square": [0, 1, 4, 9, 100],
    "is_power_of_two": [0, 1, 2, 3, 8, 32],
    "is_palindrome": [11, 22, 121, 123, 1231],
}
for prob in PROBLEMS:
    prob["probes"] = (base_probes + extra_probes[prob["name"]]
                      + list(rng.integers(5, 400, size=12)))

print("构造 %d 道 toy 题：%s" % (len(PROBLEMS), ", ".join(p["name"] for p in PROBLEMS)))
for prob in PROBLEMS:
    n_g = sum(p["good"] for p in prob["pool"])
    print("  %-15s 候选 %d（正确 %d）" % (prob["name"], len(prob["pool"]), n_g))


In [ ]:
def sample_pool(rng, problem, n, p_correct):
    """从 problem 的候选池中采样 n 个候选，p_correct 为采到正确的概率。"""
    good = [p for p in problem["pool"] if p["good"]]
    bad = [p for p in problem["pool"] if not p["good"]]
    out = []
    for _ in range(n):
        out.append(rng.choice(good) if rng.random() < p_correct else rng.choice(bad))
    return out


def simulate_pipeline(problem, rng, n_samples, p_correct, n_submit=10):
    """对一道题采样并走完整管线，返回是否解出。"""
    cands = sample_pool(rng, problem, n_samples, p_correct)
    passed = filter_by_example_tests(cands, problem["examples"])
    if not passed:
        return False
    clusters = cluster_by_signature(passed, problem["probes"])
    subs = submit_top_clusters(clusters, n_submit)
    return any(solves_hidden(c, problem["hidden"]) for c in subs)


def solve_rate(problems, rng, k, p_correct, n_trials=15):
    """多道题重复实验，返回平均解决率。"""
    hits = 0
    total = 0
    for prob in problems:
        for _ in range(n_trials):
            hits += simulate_pipeline(prob, rng, k, p_correct)
            total += 1
    return hits / total


print("sample_pool / simulate_pipeline / solve_rate 已定义")
print("k=100, p=0.2 时的解决率：%.3f"
      % solve_rate(PROBLEMS, rng, 100, 0.2, n_trials=15))


In [ ]:
import matplotlib.pyplot as plt

ks = np.array([10, 30, 100, 300, 1000], dtype=float)
pc = 0.08


def solve_rate_raw(problems, rng, k, p_correct, n_trials=20):
    """从全部采样里随机挑 10 个提交（不做过滤聚类），返回解决率。"""
    hits = 0
    total = 0
    for prob in problems:
        for _ in range(n_trials):
            cands = sample_pool(rng, prob, k, p_correct)
            rng.shuffle(cands)
            subs = cands[:10]
            hits += any(solves_hidden(c, prob["hidden"]) for c in subs)
            total += 1
    return hits / total


# 左图：无管线的随机提交 vs 过滤加聚类，以及 pass@k 上界
y_raw = np.array([solve_rate_raw(PROBLEMS, rng, int(k), pc) for k in ks])
y_pipe = np.array([solve_rate(PROBLEMS, rng, int(k), pc) for k in ks])
ypass = 1.0 - (1.0 - pc) ** ks

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(ks, y_raw, marker="o", label="random (no pipeline)")
ax1.plot(ks, y_pipe, marker="s", label="filter+cluster (10@k)")
ax1.plot(ks, ypass, marker="^", ls="--", label="pass@k (upper bound)")
ax1.set_xscale("log")
ax1.set_xlabel("number of samples k (log scale)")
ax1.set_ylabel("solve rate")
ax1.set_title("Pipeline vs raw sampling")
ax1.legend()

# 右图：解析 pass@k 的 log-linear 增长。真实竞赛题每样本正确率 p 很低。
for p in [0.002, 0.005, 0.02]:
    y = 1.0 - (1.0 - p) ** ks
    ax2.plot(ks, y, marker="o", label="p=%.3f" % p)
    slope = np.polyfit(np.log10(ks), y, 1)[0]
    print("p=%.3f 的 log-linear 拟合斜率：%.2f" % (p, slope))
ax2.set_xscale("log")
ax2.set_xlabel("number of samples k (log scale)")
ax2.set_ylabel("solve rate")
ax2.set_title("Log-linear growth of pass@k")
ax2.legend()

plt.tight_layout()
plt.show()
for p in [0.002, 0.02, 0.2]:
    k_half = np.log(0.5) / np.log(1 - p)
    print("达到 50%% 求解率所需样本：p=%.3f -> %.0f" % (p, k_half))
print("关键观察：随机提交的解决率被每样本正确率卡住，过滤加聚类逼近上界；")
print("p 越高 log-linear 曲线越靠左，达到相同求解率所需样本数越少。")


## 2. AlphaCode 2：从采样到过滤，再加重排序

AlphaCode 2 保留采样、过滤、聚类三件套，把"每簇挑一个"升级为"每簇挑最优"：再微调一个打分模型预估候选的正确性，在簇内选分数最高的作为提交。改动虽小，效果显著。下面用两种方式观察重排序的价值。先对比四种提交策略：随机挑、只过滤、过滤加聚类、过滤加聚类再加簇内打分。


In [ ]:
def score_by_simulated_model(c):
    """模拟微调打分模型：候选正确性的 0-1 预估（用 good 标记模拟）。"""
    return 1.0 if c["good"] else 0.0


def strategy_compare(problem, rng, n_samples, p_correct, n_submit=10):
    """比较四种提交策略，返回各策略的解出情况（0/1）。

    四种策略：从全部采样随机挑、过滤后随机挑、过滤加聚类挑代表、
    过滤加聚类加簇内打分。
    """
    cands = sample_pool(rng, problem, n_samples, p_correct)
    passed = filter_by_example_tests(cands, problem["examples"])
    if not passed:
        return (0, 0, 0, 0)
    shuffled = cands[:]
    rng.shuffle(shuffled)
    random_subs = shuffled[:n_submit]
    filter_subs = passed[:n_submit]
    clusters = cluster_by_signature(passed, problem["probes"])
    cluster_subs = submit_top_clusters(clusters, n_submit)
    scored_subs = [max(members, key=score_by_simulated_model)
                   for _, members in clusters[:n_submit]]
    hid = problem["hidden"]
    solved = lambda subs: any(solves_hidden(c, hid) for c in subs)
    return (solved(random_subs), solved(filter_subs),
            solved(cluster_subs), solved(scored_subs))


agg = {"random": [], "filter": [], "cluster": [], "cluster+score": []}
for prob in PROBLEMS:
    for _ in range(40):
        r_, f_, c_, cs_ = strategy_compare(prob, rng, 200, 0.08)
        agg["random"].append(r_)
        agg["filter"].append(f_)
        agg["cluster"].append(c_)
        agg["cluster+score"].append(cs_)

print("弱质量（p_correct=0.08）下四种提交策略的解决率（4 题 × 40 次实验）：")
for k, v in agg.items():
    print("  %-14s %.2f" % (k, np.mean(v)))
print("关键观察：无管线的随机提交命中率最低；过滤提升了正确候选密度；")
print("聚类覆盖不同行为簇，正确解所在簇被提交后即可解出。")


In [ ]:
# 弱模型场景：正确解不是最大簇时，聚类失效，打分模型弥补
rng2 = np.random.default_rng(7)
weak = filter_by_example_tests(
    sample_candidates(rng2, 400, p_correct=0.08), EXAMPLE_TESTS)
weak_clusters = cluster_by_signature(weak, probe_inputs)
weak_cluster_subs = submit_top_clusters(weak_clusters)
weak_score_subs = sorted(weak, key=score_by_simulated_model, reverse=True)[:10]

print("弱模型（p_correct=0.08）下过滤后剩 %d 个候选" % len(weak))
print("最大簇大小 %d，其中正确 %d 个"
      % (len(weak_clusters[0][1]), sum(c["good"] for c in weak_clusters[0][1])))
print("聚类提交解出：%d，打分提交解出：%d"
      % (sum(solves_hidden(c, HIDDEN_TESTS) for c in weak_cluster_subs),
         sum(solves_hidden(c, HIDDEN_TESTS) for c in weak_score_subs)))
print("关键观察：正确样本少时，错误解的行为簇更大，"
      "按簇大小挑会错过正确解；打分模型按正确性估计挑则不会。")


## 3. 智能体搜索增强推理：Search-o1

把搜索从程序空间搬到知识空间。长推理模型擅长逐步推理，但记忆有边界。统计显示这类模型在推理中高频出现不确定词，比如 perhaps 平均每个输出出现 30 次。这些词是知识缺口的信号：模型推理到某一步时，发现自己缺少一段事实。


In [ ]:
# 手工构造一段长推理文本，模拟 o1 类模型推理时高频出现的不确定词
chain = (
    "这个问题是物理题。wait，让我想想。perhaps 相对论在这里适用，"
    "maybe 我把公式记反了。likely 需要查洛伦兹因子，"
    "hmm 我对这一节不确定，wait 也许可以先忽略。"
    "perhaps 我还是先假设静止参照系。"
)
tokens = ["perhaps", "wait", "maybe", "likely", "hmm"]
counts = {t: chain.count(t) for t in tokens}
print("推理链中的不确定词出现次数：")
for t, c in counts.items():
    print("  %-8s %d" % (t, c))

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.5))
plt.bar(list(counts.keys()), list(counts.values()), color="#4C72B0")
plt.xlabel("uncertainty word")
plt.ylabel("count")
plt.title("Uncertainty markers in a reasoning chain")
plt.tight_layout()
plt.show()
print("关键观察：这些不确定词是推理中知识缺口的信号，Search-o1 把它们当作触发检索的线索。")


Search-o1 把处理知识缺口的工作拆给两个角色。`reasoning-actor` 决定何时搜：推理到不确定处，自行输出一个被特殊符号包裹的搜索查询，检测到结束符即暂停推理。`retrieval-critic` 决定怎么用：独立于主推理链运行，先把检索文档精炼成一小段知识，再以结果符号插回推理链，推理继续。这样在一条推理链内可以触发多轮检索，各步知识需求不同也能覆盖。

下面用一个小知识库与统一 LLM 客户端复现这个循环。


In [ ]:
# 统一 LLM 客户端：有 API key 用真实模型，否则自动进入 mock 模式
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()
print("LLM 模式：", "mock（确定性占位输出）" if client.is_mock else "real")

# 小知识库：20 条事实，key 为关键词，value 为可引用内容
KB = {
    "爱因斯坦": "爱因斯坦生于德国乌尔姆，理论物理学家。",
    "乌尔姆": "乌尔姆是德国巴登-符腾堡州的城市，多瑙河畔。",
    "德国": "德国是欧洲国家，首都是柏林。",
    "德国首都": "德国的首都是柏林。",
    "柏林": "柏林是德国首都，人口约 360 万。",
    "法国": "法国是欧洲国家，首都是巴黎。",
    "巴黎": "巴黎是法国首都，人口约 210 万。",
    "牛顿": "牛顿是英国物理学家，提出万有引力定律。",
    "引力": "万有引力定律描述质量之间的相互吸引。",
    "相对论": "相对论是爱因斯坦提出的时空理论。",
    "光速": "光速约为每秒 30 万公里。",
    "质能": "质能方程 E=mc^2 出自爱因斯坦。",
    "中国": "中国是亚洲国家，首都是北京。",
    "北京": "北京是中国的首都。",
    "日本": "日本是亚洲国家，首都是东京。",
    "东京": "东京是日本的首都，人口约 1400 万。",
    "英国": "英国是欧洲国家，首都是伦敦。",
    "伦敦": "伦敦是英国的首都，人口约 900 万。",
    "Linus": "Linus 创建了 Linux 内核，出生于芬兰。",
    "芬兰": "芬兰是北欧国家，首都是赫尔辛基。",
}


def retrieve(query, kb):
    """在知识库中按关键词检索，返回命中的 (key, value) 列表。"""
    tokens = query.split()
    hits = []
    for key, value in kb.items():
        if any(tok in key for tok in tokens):
            hits.append((key, value))
    return hits


print("检索示例 '德国 首都'：", retrieve("德国 首都", KB))


In [ ]:
SEARCH_OPEN = "<|begin_search_query|>"
SEARCH_CLOSE = "<|end_search_query|>"
RESULT_OPEN = "<|begin_search_result|>"
RESULT_CLOSE = "<|end_search_result|>"


def extract_queries(text):
    """返回文本中按出现顺序排列的搜索查询列表。"""
    queries = []
    parts = text.split(SEARCH_OPEN)
    for part in parts[1:]:
        if SEARCH_CLOSE in part:
            queries.append(part.split(SEARCH_CLOSE)[0])
    return queries


def make_reasoner(client):
    """构造 Search-o1 的 reasoning-actor：输出推理链、触发搜索查询。

    mock 模式下返回脚本化轨迹；真实模式调用底层模型。返回 (first, next_)：
    first 生成推理链开头，next_ 在检索结果注入后继续推理。
    """
    if client.is_mock:
        steps = [
            "我记得爱因斯坦出生在德国，但对具体城市拿不准。"
            "<|begin_search_query|>爱因斯坦 出生地<|end_search_query|>",
            "乌尔姆位于德国，那么德国的首都呢？"
            "<|begin_search_query|>德国 首都<|end_search_query|>",
            "综合出生地与首都信息，答案应是柏林。",
        ]
        state = {"i": 0}

        def first(question):
            return steps[0]

        def next_(context):
            state["i"] += 1
            return steps[min(state["i"], len(steps) - 1)]

        return first, next_

    def first(question):
        return client.chat([{"role": "user",
                             "content": question + " 请逐步推理，不确定处用 "
                             + SEARCH_OPEN + "查询" + SEARCH_CLOSE + "触发检索。"}])

    def next_(context):
        return client.chat([{"role": "user",
                             "content": "接续推理：" + context}])

    return first, next_


def refine_docs(chain, query, docs, client):
    """retrieval-critic：把检索文档精炼成一段可注入推理链的知识。"""
    if client.is_mock:
        if docs:
            return "根据资料：" + docs[0][1]
        return "未检索到相关信息。"
    prompt = ("当前推理链：%s\n检索查询：%s\n检索文档：%s\n"
              "请把文档精炼成简洁、可直接用于推理的知识。"
              % (chain, query, docs))
    return client.chat([{"role": "user", "content": prompt}])


In [ ]:
def run_search_o1(question, kb, client, max_rounds=3):
    """执行 Search-o1 式检索注入循环，返回 (最终推理链, 检索轮数, 查询列表)。"""
    first, next_ = make_reasoner(client)
    chain = first(question)
    rounds = 0
    processed = 0
    queries_done = []
    while rounds < max_rounds:
        queries = extract_queries(chain)
        if len(queries) <= processed:
            break
        query = queries[processed]
        processed += 1
        docs = retrieve(query, kb)
        refined = refine_docs(chain, query, docs, client)
        chain = chain + RESULT_OPEN + refined + RESULT_CLOSE
        chain = chain + next_(chain)
        queries_done.append(query)
        rounds += 1
    return chain, rounds, queries_done


question = "爱因斯坦出生于哪个国家？这个国家的首都是哪个城市？"
final_chain, n_rounds, queries = run_search_o1(question, KB, client)
print("检索轮数：%d" % n_rounds)
for i, q in enumerate(queries):
    print("  第 %d 轮查询：%s" % (i + 1, q))
print("最终推理链：")
print(final_chain)
print("关键观察：一条推理链内触发两轮检索，各步知识需求不同也能覆盖。")


`标准 RAG` 只按原始问题检索一次。多跳问题的中间实体往往不在原始问题里，一次检索覆盖不到各步不同的知识缺口。按需检索（`agentic RAG`）每步触发查询，能组合出跨越多个实体的答案。论文报告，在多跳开放域问答上，按需检索比标准 RAG 的平均 EM 提升 23.2%。


In [ ]:
def standard_rag(question, kb, client):
    """标准 RAG：只按原始问题检索一次，然后直接生成答案。"""
    docs = retrieve(question, kb)
    if client.is_mock:
        if not docs:
            msg = "mock：按原始问题检索未命中条目，无法覆盖多跳。"
        else:
            msg = "mock：只命中 %d 条，未覆盖中间实体。" % len(docs)
        return msg, docs
    prompt = "根据以下资料回答问题：%s\n问题：%s" % (docs, question)
    return client.chat([{"role": "user", "content": prompt}]), docs


std_answer, std_docs = standard_rag(question, KB, client)
_, n_agent_rounds, agent_queries = run_search_o1(question, KB, client)
print("Standard RAG：查询 1 次，命中 %d 条" % len(std_docs))
print("  " + std_answer)
print("Agentic RAG：检索 %d 轮，按序查询：%s"
      % (n_agent_rounds, " -> ".join(agent_queries)))
print("关键观察：多跳问题的中间实体不在原始问题里，"
      "一次检索覆盖不到，按需检索才能拼出完整答案。")


## 4. 深度研究的工作流

把三篇论文的组件拼成一个深度研究的最小闭环。深度研究面向一个开放问题，流程是四个环节：把复杂问题分解为若干可检索的子问题（规划），对每个子问题执行检索（agentic RAG），把证据综合成报告（综合），为报告补上来源（引用）。AlphaCode 的过滤加聚类处理的是候选程序，Search-o1 的检索注入处理的是知识条目，深研工作流把它们串成面向一个研究问题的管线。


In [ ]:
def plan_questions(topic, client):
    """把研究主题分解为可检索的子问题（规划环节）。"""
    if client.is_mock:
        return ["乌尔姆 城市", "柏林 首都", "乌尔姆 德国", "柏林 人口"]
    content = "把主题 '%s' 分解为 3-4 个可检索的子问题。" % topic
    return [line.strip() for line in client.chat([{"role": "user",
                                                   "content": content}]).splitlines()]


def synthesize_report(topic, evidence, client):
    """把各子问题的检索证据综合成报告（综合环节）。"""
    if client.is_mock:
        lines = ["# 主题：" + topic]
        for q, docs in evidence:
            for key, val in docs[:1]:
                lines.append("- %s：%s" % (key, val))
        lines.append("结论：以上条目来自知识库，真实场景由检索结果综合而成。")
        return "\n".join(lines)
    prompt = "根据以下证据写一份简洁报告：%s" % (evidence,)
    return client.chat([{"role": "user", "content": prompt}])


def build_references(evidence):
    """从证据生成引用列表（引用环节）。"""
    refs = []
    for q, docs in evidence:
        for key, val in docs:
            refs.append("%s —— %s" % (key, val))
    return refs


def deep_research(topic, kb, client):
    """mini 深度研究：规划 -> 逐子问题检索 -> 综合 -> 引用。"""
    questions = plan_questions(topic, client)
    evidence = []
    for q in questions:
        evidence.append((q, retrieve(q, kb)))
    report = synthesize_report(topic, evidence, client)
    refs = build_references(evidence)
    return report, refs


In [ ]:
topic = "德国城市：乌尔姆与柏林在地位与人口上的差异"
report, refs = deep_research(topic, KB, client)
print(report)
print("引用 %d 条：" % len(refs))
for r in refs[:8]:
    print("  [%s]" % r)
print("（mock 模式下报告为占位输出，配置 API key 后由真实模型综合生成。）")


## 小结

- 程序合成是在巨大的程序空间里搜索，采样、过滤、聚类三件套把海量候选压缩到有限提交
- 样例测试是廉价验证器，能筛掉大部分候选；过滤阶段不能用隐藏测试
- 行为签名把输出相同的候选归为一簇，正确解聚成大簇，错误解散落
- pass@k 是所有候选全提交的上界，10@k 与它的差距衡量挑选算法的损耗
- 求解率随采样数近似 log-linear 增长，模型质量越高，曲线越早抬头、所需样本越少
- AlphaCode 2 在簇内用打分模型选最优，并把基座换强，样本效率提升超过万倍
- Search-o1 用 reasoning-actor 决定何时检索、retrieval-critic 精炼检索结果
- 一次检索（标准 RAG）覆盖不到多跳问题的中间实体，按需多次检索才拼得齐答案
- 深度研究工作流 = 规划、逐子问题检索、综合、引用四个环节的串联


## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。


**作业 1：样例测试过滤**

实现 filter_by_example_tests，让样例测试把三个候选压缩到正确的那一个。f_ok 行为正确，两个错误变体分别在 n=1 与 n=2 上出错，样例测试已覆盖这两个输入。

小提示：先用 fn(x) 逐个跑输入拿输出，再与期望比较；返回全部通过者。


In [ ]:
def f_ok(n):
    if n < 2:
        return False
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def f_wrong_one(n):
    if n == 1:
        return True
    for i in range(2, n):
        if n % i == 0:
            return False
    return True


def f_wrong_two(n):
    if n % 2 == 0:
        return False
    for i in range(3, n):
        if n % i == 0:
            return False
    return True


candidates = [{"name": "ok", "fn": f_ok},
              {"name": "wrong_one", "fn": f_wrong_one},
              {"name": "wrong_two", "fn": f_wrong_two}]
tests = [(1, False), (2, True), (3, True), (4, False), (9, False)]


def filter_by_example_tests(candidates, tests):
    """返回全部通过样例测试的候选（本题填空）。"""
    keep = []
    for c in candidates:
        ok = True
        for x, want in tests:
            if bool(c["fn"](x)) != want:
                ok = False
                break
        if ok:
            keep.append(c)
    return keep


survivors = filter_by_example_tests(candidates, tests)
assert {c["name"] for c in survivors} == {"ok"}
assert len(survivors) == 1
print("过滤后保留：", [c["name"] for c in survivors])
print("通过：样例测试把三个候选压缩到唯一正确的那一个。")


**作业 2：行为签名聚类**

实现 cluster_by_signature。outputs[i] 是第 i 个程序在一组探测输入上的输出向量，把向量相同的程序归为一簇，返回按簇大小降序的簇列表。

小提示：输出元组可以直接当 dict 的 key，把相同签名的索引收集进同一个列表。


In [ ]:
outputs = [
    [1, 0, 0, 1],   # 正确程序 A
    [1, 0, 0, 1],   # 正确程序 B（与 A 行为相同）
    [0, 0, 0, 1],   # 错误程序 C
    [0, 0, 0, 0],   # 错误程序 D
]


def cluster_by_signature(outputs):
    """把输出向量相同的程序归为一簇，返回按簇大小降序的簇列表。

    每簇是程序索引的列表（本题填空）。
    """
    groups = {}
    for i, row in enumerate(outputs):
        key = tuple(row)
        groups.setdefault(key, []).append(i)
    return sorted(groups.values(), key=len, reverse=True)


clusters = cluster_by_signature(outputs)
assert len(clusters) == 3
assert clusters[0] == [0, 1]          # 两个正确程序聚成最大簇
print("簇：", clusters)
print("通过：行为相同的程序聚在一起，正确程序构成的簇最大。")


**作业 3：检索触发与结果注入**

实现 extract_queries 与 inject_results。chain 里用特殊符号包裹搜索查询，前者按出现顺序取出查询，后者把精炼结果以结果符号插回链尾。

小提示：先用 SEARCH_OPEN 切分，再取每段到 SEARCH_CLOSE 之前的部分；注入只需字符串拼接。


In [ ]:
SEARCH_OPEN = "<|begin_search_query|>"
SEARCH_CLOSE = "<|end_search_query|>"
RESULT_OPEN = "<|begin_search_result|>"
RESULT_CLOSE = "<|end_search_result|>"

chain = ("先回忆牛顿的工作。wait，不确定。"
         "<|begin_search_query|>牛顿 国籍<|end_search_query|>"
         "他是英国科学家。那么他的出生地呢？"
         "<|begin_search_query|>牛顿 出生地<|end_search_query|>")


def extract_queries(text):
    """返回文本中按序出现的查询列表（本题填空）。"""
    queries = []
    parts = text.split(SEARCH_OPEN)
    for part in parts[1:]:
        if SEARCH_CLOSE in part:
            queries.append(part.split(SEARCH_CLOSE)[0])
    return queries


def inject_results(chain, query, result):
    """把精炼结果以 RESULT_OPEN...RESULT_CLOSE 插回链尾（本题填空）。"""
    return chain + RESULT_OPEN + result + RESULT_CLOSE


queries = extract_queries(chain)
assert queries == ["牛顿 国籍", "牛顿 出生地"]
updated = inject_results(chain, queries[0], "牛顿是英国物理学家")
assert RESULT_OPEN in updated
assert updated.endswith(RESULT_CLOSE)
print("查询列表：", queries)
print("注入后链尾：", updated[-24:])
print("通过：特殊符号对的解析与注入都完整。")


## 参考资料

- Li et al., [Competition-Level Code Generation with AlphaCode](https://arxiv.org/abs/2203.07814), 2022 — 程序合成即搜索的奠基系统，采样、过滤、聚类三件套的原始出处
- DeepMind, [AlphaCode 2 Technical Report](https://storage.googleapis.com/deepmind-media/AlphaCode2/AlphaCode2_Tech_Report.pdf), 2023 — 技术报告，无 arXiv 编号；同一系统的新重排序器与更高的样本效率
- Luo et al., [Search-o1: Agentic Search-Enhanced Large Reasoning Models](https://arxiv.org/abs/2501.05366), 2025 — 长推理模型按需检索与文档精炼，reasoning-actor 与 retrieval-critic 的出处
- DeepMind, [CodeContests 数据集](https://github.com/deepmind/code_contests) — AlphaCode 的训练与评测数据集，时间切分与生成测试
- QwQ-32B-Preview, [arXiv:2412.10903](https://arxiv.org/abs/2412.10903) — Search-o1 的推理主干，开源长推理模型
- Yao et al., [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629), 2022 — "先想后做、决定调用"的 agentic 思路来源
- Asai et al., [Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection](https://arxiv.org/abs/2310.11511), 2023 — 检索必要性判断与自反思，与 Search-o1 同源
- Stanford, [CS329A 课程大纲](https://cs329a.stanford.edu/) — 本讲在课程地图中的定位
